In [ ]:
%%sql
-- ============================================================
-- Spark SQL script to build Gold Sales Dimensions
-- Target: Microsoft Fabric Lakehouse (Delta tables)
-- Author: Converted from T-SQL Stored Procedure
-- ============================================================

-- Ensure timestamps are handled in UTC (equivalent to GETDATE / SYSUTCDATETIME)
SET spark.sql.session.timeZone = 'UTC';

-- Ensure schemas exist
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

-- ============================================================
-- 1) Customer Dimension
-- Table: gold.dimsalescustomer
-- ============================================================

CREATE OR REPLACE TABLE gold.dimsalescustomer
USING DELTA
AS
SELECT 
    c.CustomerID,
    p.FirstName,
    p.LastName,
    p.preferred_email,

    -- Replace NULL values with 'Unknown'
    coalesce(p.AddressLine1, 'Unknown')      AS AddressLine1,
    coalesce(p.City, 'Unknown')              AS City,
    coalesce(p.StateProvinceCode, 'Unknown') AS StateProvince,
    coalesce(p.CountryRegionCode, 'Unknown') AS CountryRegion,

    -- Load timestamp
    current_timestamp() AS LoadDate
FROM silver.sales_customer c
LEFT JOIN gold.person_360 p
    ON c.PersonID = p.business_entity_id;

-- ============================================================
-- 2) Product Dimension
-- Table: gold.dimsalesproduct
-- ============================================================

CREATE OR REPLACE TABLE gold.dimsalesproduct
USING DELTA
AS
SELECT 
    p.ProductID,
    p.Name,
    p.ProductNumber,

    -- Convert empty strings to NULL, then default to 'Unknown'
    coalesce(nullif(p.Color, ''), 'Unknown') AS Color,

    p.StandardCost,
    p.ListPrice,

    -- Convert empty strings to NULL, then default to 'Unknown'
    coalesce(nullif(p.Size, ''), 'Unknown')  AS Size,

    -- Default numeric NULLs
    coalesce(p.Weight, 0.0)                  AS Weight,

    sc.Name AS Subcategory,
    pc.Name AS Category,

    -- Load timestamp
    current_timestamp() AS LoadDate
FROM silver.production_product p
LEFT JOIN silver.production_productsubcategory sc
    ON p.ProductSubcategoryID = sc.ProductSubcategoryID
LEFT JOIN silver.production_productcategory pc
    ON sc.ProductCategoryID = pc.ProductCategoryID;

-- ============================================================
-- 3) Sales Territory Dimension
-- Table: gold.dimsalesterritory
-- ============================================================

CREATE OR REPLACE TABLE gold.dimsalesterritory
USING DELTA
AS
SELECT 
    TerritoryID,
    Name,
    CountryRegionCode,

    -- Group is a reserved keyword in Spark SQL, use backticks
    `Group` AS TerritoryGroup,

    -- Load timestamp
    current_timestamp() AS LoadDate
FROM silver.sales_salesterritory;

-- ============================================================
-- End of script
-- ============================================================
